# RoofTop Regression Scoping

## Purpose

Scope the APTs (Address Points) that were **relocated for ESP** so they can be validated
and, once relocation is confirmed, fed into the Data Correctness (relocation) process.

### Background (SEACO-6171)

In Q1–Q2 2026, a total of **8,569,598 (~8.5M)** APTs were ingested across multiple sources
(e.g. `CATASTRO_APT`), with locations updated according to each respective source. Some of
these ingestions moved existing APTs (originally from Genesis, Nomecalles Opensource,
APT Eustat, esp event_lost) away from their rooftop, causing a regression in the ESP RAC
(Rooftop Accuracy) metrics.

### Scope of this notebook

As the next step in the investigation, this notebook:

1. Determines the exact count of APTs that moved **outside the Building Footprint (BFP)**
   in Q1 and Q2 for ESP.
2. Scopes those relocated APTs.
3. Prepares the scoped set for validation, after which the Data Correctness process for
   relocation can be triggered.

Reference: [SEACO-6171](https://tomtom.atlassian.net/browse/SEACO-6171?focusedCommentId=12144061)

## Approach

The relocated-APT scoping is built step by step from two snapshots (OLD vs LATEST) and the
Building Footprint (BFP) layer. The BFP intersection is computed **per snapshot** (on OLD and
on LATEST independently) and only then are the two sides joined:

1. **Load the OLD snapshot** into a Delta table (pin the older revision to compare against).
2. **Load the LATEST snapshot** into a Delta table (resolve the newest revision).
3. **Read the OLD snapshot** Delta table as a `Dataset[OrbisElement]`.
4. **Read the LATEST snapshot** Delta table as a `Dataset[OrbisElement]`.
5. **Filter both snapshots by `countryISO` = `ESP`** (value of the `metadata:country` tag).
6. **Filter the OLD snapshot by `metadata:updated`** — keep only APTs whose `metadata:updated`
   value (an epoch timestamp) is **greater than 1st Feb 2026** (the regression window start).
7. **Load the Building Footprint (BFP) snapshot** (MCR `preprocess_prod.bfp`, filtered to the
   ESP license zone) and build its geometry.
8. **Enrich each snapshot with its BFP match.** For BOTH the OLD and the LATEST dataset,
   spatially join the APT point against the BFP polygons (`ST_Intersects`) and add, per APT:
   - `is_inside_bfp` — `true` when the APT point falls inside a building footprint;
   - `bfp_id` and `bfp_wkt` — the id and geometry (WKT) of the matched footprint (null when no match).
9. **Left join** the enriched OLD snapshot with the enriched LATEST snapshot on the APT id
   (keep all OLD rows; LATEST columns are null where the APT is missing in the latest).
10. **Add geometry / distance columns** on the joined dataset:
    - `old_wkt` and `latest_wkt` — `POINT(lng lat)` built from each snapshot's lat/lng;
    - `distance_in_meters` — great-circle (Haversine) distance between the OLD and LATEST point;
    - `hookline` — `LINESTRING(old -> latest)` connecting the two points.
11. **Final dataset** contains, per APT:
    - OLD snapshot details (location, tags, `is_inside_bfp`, `bfp_id`, `bfp_wkt`),
    - LATEST snapshot details (location, tags, `is_inside_bfp`, `bfp_id`, `bfp_wkt`),
    - `distance_in_meters` and `hookline`.

In [0]:
%sql
-- Drop the scoping databases if they already exist, then recreate them fresh.
-- CASCADE removes any tables left from a previous run. Run this once before the loader cells.
DROP DATABASE IF EXISTS rooftop_regression_scoping_db_old CASCADE;
DROP DATABASE IF EXISTS rooftop_regression_scoping_db_new CASCADE;
CREATE DATABASE IF NOT EXISTS rooftop_regression_scoping_db_old;
CREATE DATABASE IF NOT EXISTS rooftop_regression_scoping_db_new;

In [0]:
%scala
// Shared constants used across the cells below (defined once here; values do not change between runs).
val LAYER_ID = 14533
val COUNTRY_ISO = "ESP"
val OLD_DATABASE = "rooftop_regression_scoping_db_old"
val NEW_DATABASE = "rooftop_regression_scoping_db_new"
val oldSnapshotTable = s"${OLD_DATABASE}.apt_snapshot_old"
val latestSnapshotTable = s"${NEW_DATABASE}.apt_snapshot_new"

In [0]:
%scala
// Step 1 - Load the OLD APT snapshot (layer 14533) into a Delta table (country = ESP).
// Pin OLD_REVISION to the older revision you want to compare the latest snapshot against.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

// >>> Set the OLD snapshot revision id here <<<
val OLD_REVISION = 42394153L

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

println(s"Loading OLD snapshot revision: $OLD_REVISION")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, OLD_REVISION)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(OLD_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded old snapshot, keeping only APTs whose metadata:country tag = ESP
val oldAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
oldAptDS.write.format("delta").mode("overwrite").saveAsTable(oldSnapshotTable)
println(s"OLD snapshot written to: $oldSnapshotTable")
// display(oldAptDS)

In [0]:
%scala
// Step 2 - Load the LATEST APT snapshot (layer 14533) into a Delta table (country = ESP).
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = ESP
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")
// display(latestAptDS)

In [0]:
%scala
// Step 3 - Load the LATEST road-element snapshot (layer 6) into the NEW database.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(6)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(6, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

In [0]:
%scala
// Step 4 - Read the OLD snapshot as Dataset[OrbisElement] and enrich (country, orbisIdString, updated_epoch).
// Enrich each row with country, the unsigned colon-separated orbisIdString, and metadata:updated (epoch).

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.{Id, OrbisElement}
import java.lang.{Long => JLong}

// Build the colon-separated unsigned id string: {layerId}:{unsignedHigh}:{unsignedLow}
def convertOrbisIdToString(orbisId: Id): String = {
  val COLON_SEPARATOR = ":"
  Seq(orbisId.layerId.getOrElse(14533).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString(COLON_SEPARATOR)
}
val convertOrbisIdUDF = udf((orbisId: Id) => convertOrbisIdToString(orbisId))

val countryTagKey = "metadata:country"
val updatedTagKey = "metadata:updated"

val oldSnapshotDS: Dataset[OrbisElement] =
  spark.table(oldSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val oldSnapshotEnriched = oldSnapshotDS
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))
  // metadata:updated value is an epoch timestamp string -> cast to long for the date filter in Step 6
  .withColumn("updated_epoch",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === updatedTagKey)
      .getItem(0).getField("value").cast("long"))

println(s"OLD snapshot read from: $oldSnapshotTable, count: ${oldSnapshotEnriched.count()}")
//display(oldSnapshotEnriched)

In [0]:
%scala
// Step 5 - Read the LATEST snapshot as Dataset[OrbisElement] and enrich (country, orbisIdString).
// Enrich each row with country and the unsigned colon-separated orbisIdString (UDF defined in Step 3).

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.OrbisElement

val latestSnapshotDS: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val latestSnapshotEnriched = latestSnapshotDS
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"LATEST snapshot read from: $latestSnapshotTable, count: ${latestSnapshotEnriched.count()}")
// display(latestSnapshotEnriched)

In [0]:
%scala
// Step 6 - Filter both snapshots to countryISO = ESP.
// (Snapshots are already loaded ESP-only in Steps 1-2; this re-applies the filter defensively.)

import org.apache.spark.sql.functions._

val oldEsp = oldSnapshotEnriched.filter(col("country") === COUNTRY_ISO)
val latestEsp = latestSnapshotEnriched.filter(col("country") === COUNTRY_ISO)

println(s"OLD ESP count: ${oldEsp.count()}")
println(s"LATEST ESP count: ${latestEsp.count()}")

In [0]:
%scala
// Step 7 - Filter the OLD snapshot to APTs updated after 1st Feb 2026 (regression window start).
// metadata:updated is an epoch timestamp in SECONDS; adjust the constant if your values are in millis.

import org.apache.spark.sql.functions._

// 2026-02-01T00:00:00Z in epoch SECONDS. Use 1769904000000L if metadata:updated is in milliseconds.
val FEB_FIRST_EPOCH = 1769904000L

val oldEspUpdated = oldEsp.filter(col("updated_epoch") > FEB_FIRST_EPOCH)

println(s"OLD ESP updated after 1st Feb count: ${oldEspUpdated.count()}")
// display(oldEspUpdated)

In [0]:
%scala
// Step 8 (optional) - Refresh the MCR Building Footprint Delta table (commented out; normally pre-loaded).
// The loader writes BFP to <deltaConfig database>.bfp; in PROD that is preprocess_prod.bfp (read next cell).

/*
import org.apache.spark.sql.{Dataset, SparkSession}
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadDataFromMcr
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader

val env = "PROD" // dbutils.widgets.get("environment").toUpperCase()
println(s"Environment Name $env")

val config = SourceConfigLoader.getConfig

// deltaConfig resolves to the source_data table for the env; its database (preprocess_prod in PROD)
// is where LoadDataFromMcr persists the BFP table -> preprocess_prod.bfp
val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(bfpDatabaseName, tableName) = deltaConfig.split("\\.")
println(s"BFP database: $bfpDatabaseName")

implicit val sparky = spark
implicit val envConfig = ConfigLoader.forEnvironment(env.toLowerCase)
SparkHelper.init(bfpDatabaseName)

new LoadDataFromMcr().run()
*/

In [ ]:
%scala
// Step 9 - Read the loaded road-element layer (layer 6) from the NEW database (wkt not null).

import org.apache.spark.sql.functions._

val roadElementDataset = spark.table(s"${NEW_DATABASE}.layer_6")
  .filter(col("wkt").isNotNull)

println(s"Road element (layer_6) rows with wkt: ${roadElementDataset.count()}")
display(roadElementDataset)

In [0]:
%scala
// Step 10 - Read the ESP Building Footprint (BFP) polygons from preprocess_prod.bfp and build geometry.
// Filter to the ESP license zone and build the polygon geometry for the spatial join.

import org.apache.spark.sql.functions._
import org.apache.sedona.spark.SedonaContext

// Sedona is required for the ST_Intersects spatial join in the next cell
val sedona = SedonaContext.create(spark)

val espBfpDataset = spark.table("preprocess_prod.bfp")
  .filter(col("licenseZone") === "ESP")
  .filter(col("wkt").isNotNull)
  .withColumn("bfp_geom", expr("ST_GeomFromWKT(wkt)"))

println(s"ESP BFP polygon count: ${espBfpDataset.count()}")
display(espBfpDataset)

In [0]:
%scala
// Step 11 - Enrich BOTH snapshots with their BFP match (is_inside_bfp, bfp_id, bfp_wkt).
// An INNER ST_Intersects join (Sedona plans a partitioned RangeJoin, no broadcast) finds the
// matched footprint; a cheap equi LEFT join back onto the full APT set then yields
// is_inside_bfp, bfp_id and bfp_wkt without dropping APTs that fall outside every footprint.

import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions._

// A spatial LEFT join is NOT optimized by Sedona -> Spark uses a BroadcastNestedLoopJoin and
// broadcasts the whole ~6 GB BFP relation (EXECUTOR_BROADCAST_JOIN_OOM). Disable the broadcast
// threshold so the inner spatial join below is planned as a Sedona partitioned RangeJoin.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

// BFP polygons prepared once for the spatial join
val bfpForJoin = espBfpDataset
  .select(col("bfpId").as("bfp_id"), col("wkt").as("bfp_wkt"), col("bfp_geom"))

// For one snapshot: find the (single) intersecting footprint per APT via an inner spatial join,
// then left-join it back onto every APT so non-matching APTs are kept with is_inside_bfp = false.
def enrichWithBfp(apts: DataFrame): DataFrame = {
  val aptCols = apts.select(col("orbisIdString"), col("lat"), col("lng"), col("tags"))

  val aptPoints = aptCols
    .withColumn("apt_geom", expr("ST_GeomFromText(concat('POINT(', lng, ' ', lat, ')'))"))

  // INNER spatial join -> Sedona RangeJoin (no broadcast). Keep one footprint per APT.
  val matched = aptPoints.alias("apt")
    .join(bfpForJoin.alias("bfp"), expr("ST_Intersects(bfp.bfp_geom, apt.apt_geom)"))
    .select(
      col("apt.orbisIdString").as("orbisIdString"),
      col("bfp.bfp_id").as("bfp_id"),
      col("bfp.bfp_wkt").as("bfp_wkt"))
    .dropDuplicates("orbisIdString")

  // Equi LEFT join back onto the full APT set (no spatial predicate -> cheap shuffle join)
  aptCols
    .join(matched, Seq("orbisIdString"), "left")
    .withColumn("is_inside_bfp", col("bfp_id").isNotNull)
}

val oldEnriched = enrichWithBfp(oldEspUpdated)
val latestEnriched = enrichWithBfp(latestEsp)

println(s"OLD enriched: ${oldEnriched.count()}, inside BFP: ${oldEnriched.filter(col("is_inside_bfp")).count()}")
println(s"LATEST enriched: ${latestEnriched.count()}, inside BFP: ${latestEnriched.filter(col("is_inside_bfp")).count()}")
//display(oldEnriched)

In [ ]:
%scala
// Step 12 - Join OLD vs LATEST on orbisIdString; add distance_in_meters and hookline.
// old_wkt / latest_wkt (POINT(lng lat)), distance_in_meters (Haversine) and hookline (LINESTRING old->latest).

import org.apache.spark.sql.functions._

val oldRenamed = oldEnriched
  .withColumn("old_wkt", concat(lit("POINT("), col("lng"), lit(" "), col("lat"), lit(")")))
  .withColumnRenamed("lat", "old_lat")
  .withColumnRenamed("lng", "old_lon")
  .withColumnRenamed("tags", "old_tags")
  .withColumnRenamed("is_inside_bfp", "old_is_inside_bfp")
  .withColumnRenamed("bfp_id", "old_bfp_id")
  .withColumnRenamed("bfp_wkt", "old_bfp_wkt")

val latestRenamed = latestEnriched
  .withColumn("latest_wkt", concat(lit("POINT("), col("lng"), lit(" "), col("lat"), lit(")")))
  .withColumnRenamed("lat", "latest_lat")
  .withColumnRenamed("lng", "latest_lon")
  .withColumnRenamed("tags", "latest_tags")
  .withColumnRenamed("is_inside_bfp", "latest_is_inside_bfp")
  .withColumnRenamed("bfp_id", "latest_bfp_id")
  .withColumnRenamed("bfp_wkt", "latest_bfp_wkt")

// Haversine distance in meters (mean earth radius) - avoids Sedona for the distance calc
val EARTH_RADIUS_M = 6371008.8
val dLat = radians(col("latest_lat") - col("old_lat"))
val dLon = radians(col("latest_lon") - col("old_lon"))
val haversine =
  pow(sin(dLat / 2), 2) +
  cos(radians(col("old_lat"))) * cos(radians(col("latest_lat"))) * pow(sin(dLon / 2), 2)
val distanceMeters = lit(2 * EARTH_RADIUS_M) * asin(sqrt(haversine))

val oldVsLatest = oldRenamed
  .join(latestRenamed, Seq("orbisIdString"), "left")
  .withColumn("distance_in_meters",
    when(col("old_wkt").isNotNull && col("latest_wkt").isNotNull, distanceMeters))
  .withColumn("hookline",
    when(col("old_wkt").isNotNull && col("latest_wkt").isNotNull,
      concat(lit("LINESTRING("),
        col("old_lon"), lit(" "), col("old_lat"), lit(", "),
        col("latest_lon"), lit(" "), col("latest_lat"), lit(")"))))

println(s"oldVsLatest rows where distance_in_meters != 0: ${oldVsLatest.filter(col("distance_in_meters") =!= 0).count()}")

In [ ]:
%scala
// Step 13 - Assemble & filter the scope, flag road crossings, and write the three CSVs.
// Keep only the regression candidates: APT was inside a BFP in OLD, is outside in LATEST,
// and the latest edit was NOT made by VERTEX (latest_tags metadata:user != VERTEX).
// Three CSVs are written: one WITH the tag columns, one WITHOUT, and the DC relocation input format.

import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions._

// Never auto-broadcast the large spatial relations (BFP / road layer) — keep this set here too so
// the joins below don't depend on an earlier cell having run (avoids EXECUTOR_BROADCAST_JOIN_OOM).
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

val finalScopedBase = oldVsLatest
  .select(
    col("orbisIdString"),
    col("old_lat"), col("old_lon"), col("old_wkt"), col("old_tags"),
    col("old_is_inside_bfp"), col("old_bfp_id"), col("old_bfp_wkt"),
    col("latest_lat"), col("latest_lon"), col("latest_wkt"), col("latest_tags"),
    col("latest_is_inside_bfp"), col("latest_bfp_id"), col("latest_bfp_wkt"),
    col("distance_in_meters"),
    col("hookline"))
  // Regression scope: inside BFP before, outside BFP after, and the latest edit
  // was NOT made by VERTEX (latest_tags has no metadata:user = VERTEX)
  .filter(col("old_is_inside_bfp") === true
    && col("latest_is_inside_bfp") === false
    && !exists(col("latest_tags"), t =>
      t.getField("tagKey").getField("key") === "metadata:user"
      && t.getField("value") === "VERTEX"))
  // Flag whether the APT carries metadata:apa:improvement = yes in old_tags
  .withColumn("is_APA_Improvement_exist",
    exists(col("old_tags"), t =>
      t.getField("tagKey").getField("key") === "metadata:apa:improvement"
      && t.getField("value") === "yes"))
  // The scope is reused by both the road-intersection join and the downstream writes, so cache it.
  .cache()

// Flag whether the APT's hookline (old -> latest line) intersects any road element (layer 6) and,
// when it does, capture the colon-separated road id(s) it crosses (via convertOrbisIdUDF from Step 3).
// The scoped hookline set is tiny vs the ~6 GB road layer, so BROADCAST the small (hookline) side:
// this makes Sedona plan a BroadcastIndexJoin (spatial index built on the broadcast side) and
// guarantees the large road relation is never broadcast (the cause of EXECUTOR_BROADCAST_JOIN_OOM).
// A hookline can cross several roads, so the matched road ids are collected into one comma-separated
// string per APT; a LEFT join back onto the full scope then yields the flag (false where none cross).
// Keep only drivable/road highway types so footways etc. don't count as a road crossing.
val includedHighwayValues = Seq(
  "motorway_link", "motorway", "living_street", "pedestrian", "primary", "primary_link",
  "residential", "secondary", "secondary_link", "tertiary", "tertiary_link", "track",
  "trunk", "trunk_link", "unclassified", "service")

val roadGeomForJoin = roadElementDataset
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "highway"
    && t.getField("value").isin(includedHighwayValues: _*)))
  .withColumn("road_geom", expr("ST_GeomFromWKT(wkt)"))
  .withColumn("road_id_string", convertOrbisIdUDF(col("id")))
  .select(col("road_id_string"), col("road_geom"))

val scopedHooklines = finalScopedBase
  .filter(col("hookline").isNotNull)
  .select(col("orbisIdString"), expr("ST_GeomFromText(hookline)").as("hookline_geom"))

val roadIntersected = roadGeomForJoin.alias("road")
  .join(broadcast(scopedHooklines).alias("apt"),
    expr("ST_Intersects(road.road_geom, apt.hookline_geom)"))
  .select(col("apt.orbisIdString").as("orbisIdString"), col("road.road_id_string").as("road_id_string"))
  .groupBy("orbisIdString")
  .agg(concat_ws(",", collect_set(col("road_id_string"))).as("intersecting_road_element_ids"))
  .withColumn("is_intersects_with_road_element", lit(true))

val finalScoped = finalScopedBase
  .join(roadIntersected, Seq("orbisIdString"), "left")
  .withColumn("is_intersects_with_road_element",
    coalesce(col("is_intersects_with_road_element"), lit(false)))

val outputDir = "/mnt/aqua/data-correctness/correction-input/SEACO-6182-ESP-Relocation-Scope"

// Write a DataFrame as one named .csv file: Spark emits a folder of part files, so coalesce to a
// single partition, then promote that part file to the target name and clean up the temp dir.
def writeSingleCsv(df: DataFrame, fileName: String): Unit = {
  val target = s"$outputDir/$fileName"
  val tmpDir = s"$outputDir/${fileName}_tmp"
  df.coalesce(1).write.option("header", "true").mode("overwrite").csv(tmpDir)
  val partFile = dbutils.fs.ls(tmpDir).filter(_.name.endsWith(".csv")).head.path
  dbutils.fs.rm(target)
  dbutils.fs.cp(partFile, target)
  dbutils.fs.rm(tmpDir, recurse = true)
  println(s"CSV written to: $target")
}

// WITH tags - keep only rows where is_APA_Improvement_exist is false and the hookline does NOT
// cross a road element; CSV cannot serialize array<struct>, so emit the tag columns as JSON strings.
val finalScopedWithTags = finalScoped
  .filter(col("is_APA_Improvement_exist") === false
    && col("is_intersects_with_road_element") === false)
  .withColumn("old_tags", to_json(col("old_tags")))
  .withColumn("latest_tags", to_json(col("latest_tags")))

// WITHOUT tags - keep only rows where is_APA_Improvement_exist is false, the APT moved less than
// 30 meters, and the hookline does NOT cross a road element; drop both tag columns.
val finalScopedNoTags = finalScoped
  .filter(col("is_APA_Improvement_exist") === false
    && col("distance_in_meters") < 30
    && col("is_intersects_with_road_element") === false)
  .drop("old_tags", "latest_tags")

// DC relocation input format - derived from the no-tags scope: keep every existing column and
// append the columns the Data Correctness relocation process expects.
val finalRelocationScope = finalScopedNoTags
  // Product_orbis_id: orbisIdString with ':' replaced by '_' (14533:hi:lo -> 14533_hi_lo)
  .withColumn("Product_orbis_id", regexp_replace(col("orbisIdString"), ":", "_"))
  // Address component being corrected and its target (latest) location
  .withColumn("Product_Address_component", lit("metadata:apa:previous_location"))
  .withColumn("target_value", col("latest_wkt"))
  // Orbis_X / Orbis_Y parsed from old_wkt = POINT(X Y): X is the first coord (lon), Y the second (lat)
  .withColumn("Orbis_Y", regexp_extract(col("old_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 1))
  .withColumn("Orbis_X", regexp_extract(col("old_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 2))
  .withColumn("Relocation_Type", lit("BUILDING_FOOTPRINT"))

writeSingleCsv(finalScopedWithTags, "SEACO-6182-ESP-Relocation-Scope-with-tags-without-road-crossed-filtered-footway.csv")
writeSingleCsv(finalScopedNoTags, "SEACO-6182-ESP-Relocation-Scope-without-road-crossed-filtered-footway.csv")
writeSingleCsv(finalRelocationScope, "SEACO-6182-ESP-Relocation-DC-Input-format-without-road-crossed-filtered-footway.csv")

println(s"Final scoped APT count: ${finalScoped.count()}")
println(s"finalScopedWithTags count: ${finalScopedWithTags.count()}")
println(s"finalScopedNoTags count: ${finalScopedNoTags.count()}")
println(s"finalRelocationScope count: ${finalRelocationScope.count()}")

In [ ]:
%scala
// Step 14 - Count scoped APTs by distance_in_meters band.

import org.apache.spark.sql.functions._

val distanceGrouped = finalScoped
  .withColumn("distance_group",
    when(col("distance_in_meters") <= 10, "0-10 meters")
      .when(col("distance_in_meters") <= 20, "10-20 meters")
      .when(col("distance_in_meters") <= 30, "20-30 meters")
      .when(col("distance_in_meters") <= 40, "30-40 meters")
      .when(col("distance_in_meters") <= 50, "40-50 meters")
      .when(col("distance_in_meters") <= 100, "50-100 meters")
      .when(col("distance_in_meters") <= 125, "100-125 meters")
      .when(col("distance_in_meters") <= 150, "125-150 meters")
      .when(col("distance_in_meters") <= 200, "150-200 meters")
      .when(col("distance_in_meters") <= 300, "200-300 meters")
      .when(col("distance_in_meters") <= 500, "300-500 meters")
      .when(col("distance_in_meters") <= 1000, "500-1000 meters")
      .otherwise("1000+ meters"))
  // group_order keeps the bands in ascending distance order for display
  .withColumn("group_order",
    when(col("distance_in_meters") <= 10, 1)
      .when(col("distance_in_meters") <= 20, 2)
      .when(col("distance_in_meters") <= 30, 3)
      .when(col("distance_in_meters") <= 40, 4)
      .when(col("distance_in_meters") <= 50, 5)
      .when(col("distance_in_meters") <= 100, 6)
      .when(col("distance_in_meters") <= 125, 7)
      .when(col("distance_in_meters") <= 150, 8)
      .when(col("distance_in_meters") <= 200, 9)
      .when(col("distance_in_meters") <= 300, 10)
      .when(col("distance_in_meters") <= 500, 11)
      .when(col("distance_in_meters") <= 1000, 12)
      .otherwise(13))
  .groupBy("group_order", "distance_group")
  .agg(count("*").as("apt_count"))
  .orderBy("group_order")
  .drop("group_order")

display(distanceGrouped)

In [ ]:
%scala
// Step 15 - Count scoped APTs by distance band, broken down by processingType.

import org.apache.spark.sql.functions._

val distanceProcessingTypeGrouped = finalScoped
  .withColumn("processingType",
    filter(col("latest_tags"), t => t.getField("tagKey").getField("key") === "metadata:user")
      .getItem(0).getField("value"))
  .withColumn("distance_group",
    when(col("distance_in_meters") <= 10, "0-10 meters")
      .when(col("distance_in_meters") <= 20, "10-20 meters")
      .when(col("distance_in_meters") <= 30, "20-30 meters")
      .when(col("distance_in_meters") <= 40, "30-40 meters")
      .when(col("distance_in_meters") <= 50, "40-50 meters")
      .when(col("distance_in_meters") <= 100, "50-100 meters")
      .when(col("distance_in_meters") <= 125, "100-125 meters")
      .when(col("distance_in_meters") <= 150, "125-150 meters")
      .when(col("distance_in_meters") <= 200, "150-200 meters")
      .when(col("distance_in_meters") <= 300, "200-300 meters")
      .when(col("distance_in_meters") <= 500, "300-500 meters")
      .when(col("distance_in_meters") <= 1000, "500-1000 meters")
      .otherwise("1000+ meters"))
  // group_order keeps the bands in ascending distance order for display
  .withColumn("group_order",
    when(col("distance_in_meters") <= 10, 1)
      .when(col("distance_in_meters") <= 20, 2)
      .when(col("distance_in_meters") <= 30, 3)
      .when(col("distance_in_meters") <= 40, 4)
      .when(col("distance_in_meters") <= 50, 5)
      .when(col("distance_in_meters") <= 100, 6)
      .when(col("distance_in_meters") <= 125, 7)
      .when(col("distance_in_meters") <= 150, 8)
      .when(col("distance_in_meters") <= 200, 9)
      .when(col("distance_in_meters") <= 300, 10)
      .when(col("distance_in_meters") <= 500, 11)
      .when(col("distance_in_meters") <= 1000, 12)
      .otherwise(13))
  .groupBy("group_order", "distance_group", "processingType")
  .agg(count("*").as("apt_count"))
  .orderBy("group_order", "processingType")
  .drop("group_order")

display(distanceProcessingTypeGrouped)

In [0]:
%scala
// Step 16 - Display the final scoped relocated-APT dataset.
display(finalScoped)

In [0]:
%scala
// Step 17 - Display scoped records that moved more than 125 meters.
import org.apache.spark.sql.functions._

display(finalScoped.filter(col("distance_in_meters") > 125))

In [0]:
%scala
// Step 18 - Count scoped APTs carrying metadata:apa:improvement = yes.
import org.apache.spark.sql.functions._

val apaImprovementCount = finalScoped.filter(col("is_APA_Improvement_exist")).count()
println(s"Scoped APTs with metadata:apa:improvement=yes (is_APA_Improvement_exist=true): $apaImprovementCount")

In [0]:
%scala
// Step 19 - Display scoped rows where is_APA_Improvement_exist is true.
import org.apache.spark.sql.functions._

display(finalScoped.filter(col("is_APA_Improvement_exist")))

In [ ]:
%scala
// Step 20 - Exclude already-scoped APTs (from a prior DC-input CSV) via left_anti join.
import org.apache.spark.sql.functions._

val alreadyScopedCsvPath =
  "/mnt/aqua/data-correctness/correction-input/SEACO-6182-ESP-Relocation-Scope/SEACO-6182-ESP-Relocation-DC-Input-format-without-road-crossed.csv"

val alreadyScopedApts = spark.read.option("header", "true").csv(alreadyScopedCsvPath)
  .select(col("orbisIdString"))
  .distinct()

val finalRelocationScopeExcludingAlreadyScoped = finalRelocationScope
  .join(alreadyScopedApts, Seq("orbisIdString"), "left_anti")

println(s"finalRelocationScope: ${finalRelocationScope.count()}, " +
  s"in CSV: ${alreadyScopedApts.count()}, " +
  s"remaining after exclusion: ${finalRelocationScopeExcludingAlreadyScoped.count()}")

display(finalRelocationScopeExcludingAlreadyScoped)